# BioImage Archive submission from an OMERO annotation table

Select an annotation table on OMERO, pull its metadata and data down, and package a
**BioImage Archive (BIA)** submission bundle: MIFA metadata (`Study`/`Annotations`/`Version`
YAML), the BIA file lists (`file_list_images.tsv`, `file_list_annotations.tsv`), and the
image/mask files, all in one self-contained folder.

Same starting point as the training notebooks — pick a table — but instead of training a
model, this prepares an archive submission. It is **read-only against OMERO**: nothing is
written back to the server.

For the offline variant (data already on this machine from an annotation run), see
`prepare_bioimage_archive.ipynb`.

> Requires the optional `bia-mifa-models` package (in the `dev`/`mifa` pixi environments).

## 1. Setup

In [ ]:
import datetime
from pathlib import Path

import pandas as pd

from omero_annotate_ai import (
    create_omero_connection_widget,
    create_training_data_widget,
    create_default_config,
    prepare_bia_data_from_table,
)
from omero_annotate_ai.core.annotation_config import AuthorInfo
from omero_annotate_ai.omero.omero_functions import (
    download_annotation_config_from_omero,
    sync_omero_table_to_config,
)

## 2. OMERO connection

In [ ]:
conn_widget = create_omero_connection_widget()
conn_widget.display()

In [ ]:
conn = conn_widget.get_connection()

if conn is None:
    raise ConnectionError("No OMERO connection established.")

print(f"Connected to OMERO as: {conn.getUser().getName()}")

## 3. Select the annotation table

Pick the container, scan it for annotation tables, and select the one to publish.

In [ ]:
table_widget = create_training_data_widget(connection=conn)
table_widget.display()

In [ ]:
table_id = table_widget.get_selected_table_id()
table_info = table_widget.get_selected_table_info()
container = table_widget.get_selected_container()

if not table_id:
    raise ValueError("No annotation table selected. Please select a table above.")

table_name = table_info.get("name", f"table_{table_id}")
print(f"Selected table: {table_name} (ID: {table_id})")
print(f"From container: {container['type']} {container['id']}")

## 4. Fetch the annotation config from OMERO

The annotation pipeline attaches its `AnnotationConfig` YAML to the container it ran on, so
the study metadata (title, authors, license, methodology) usually already lives on OMERO.
We load that, then fill in the table's rows as the config's annotations.

If no config is attached — e.g. the table came from somewhere else — we fall back to a
default config, and you fill in the metadata by hand in the next cell.

In [ ]:
# OMERO object types are capitalized ('Dataset'); the widget reports them lowercase.
config = download_annotation_config_from_omero(
    conn, container["type"].capitalize(), container["id"]
)

if config is None:
    print("No annotation config attached to this container - starting from a default config.")
    print("Fill in the study metadata in the next cell.")
    config = create_default_config()
    config.name = table_name
else:
    print(f"Loaded config '{config.name}' from {container['type']} {container['id']}")

# Table rows -> config.annotations
sync_omero_table_to_config(conn, table_id, config)
print(f"Annotations in config: {len(config.annotations)}")

## 5. Review and complete the study metadata

The BioImage Archive requires a title, description, license, authors and a funding statement.
The values below are prefilled from the config that came off OMERO — edit what is missing or
wrong. Anything you leave unset falls back to a placeholder in the MIFA metadata, which is
usually not what you want in a public submission.

In [ ]:
# --- edit these -------------------------------------------------------------
config.study.title = config.study.title or "Nuclei segmentation in U2OS cells"
config.study.description = config.study.description or "Expert-curated nuclear segmentation masks."
config.study.keywords = config.study.keywords or ["segmentation", "fluorescence"]
config.study.organism = config.study.organism or "Homo sapiens"
config.study.funding_statement = (
    config.study.funding_statement or "Supported by the Example Funding Council (grant 12345)."
)
config.dataset.license = config.dataset.license or "CC-BY-4.0"  # MIFA allows CC0 or CC-BY only

if not config.authors:
    config.authors = [AuthorInfo(name="Jane Doe", affiliation="Your Institution")]

# Accession: use the real S-BIAD id once the BIA has issued one; until then this is just the
# name stamped into the MIFA filenames.
accession = config.dataset.source_dataset_id or "S-BIADXXX"
# ----------------------------------------------------------------------------

print(f"Title:    {config.study.title}")
print(f"License:  {config.dataset.license}")
print(f"Authors:  {', '.join(a.name for a in config.authors)}")
print(f"Funding:  {config.study.funding_statement}")
print(f"Accession: {accession}")

## 6. Download the images and masks from OMERO

Pulls each annotated plane/patch and its mask into the layout the bundle builder expects
(`input/` + `output/`). Images are written at their **native bit depth** — unlike the training
data prep, which normalizes to 8-bit, an archive submission keeps the original pixel values.

Rows that were never processed, or that carry no mask, are skipped and reported.

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
work_dir = Path.home() / "omero-annotate-ai" / "bia-export" / f"{table_name}_{timestamp}"

stats = prepare_bia_data_from_table(
    conn=conn,
    table_id=table_id,
    output_dir=work_dir,
    config=config,
)

# The bundle builder copies from here.
config.output.output_directory = work_dir

print(f"Downloaded to: {work_dir}")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 7. Build the BIA submission bundle

In [ ]:
bundle_dir = work_dir / "submission"
result = config.save_bia_package(bundle_dir, accession=accession)

print("Bundle written to:", result["dir"])
print(
    f"  images: {result['n_images']} | annotations: {result['n_annotations']} "
    f"| files copied: {result['copied']} | missing: {result['missing']}"
)
print("\nBundle contents:")
for path in sorted(result["dir"].rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(result["dir"]))

## 8. Inspect the file lists and MIFA metadata

Check these before submitting. Every row of `file_list_annotations.tsv` must point at a real
mask file, and its `source_image` at the raw image that mask belongs to.

In [ ]:
print("=== file_list_annotations.tsv ===")
print(pd.read_csv(result["file_list_annotations"], sep="\t").to_string(index=False))
print("\n=== file_list_images.tsv ===")
print(pd.read_csv(result["file_list_images"], sep="\t").to_string(index=False))
print(f"\n=== metadata/Annotations_{accession}.yaml ===")
print(result["metadata"]["annotations"].read_text())

## 9. Upload to the BioImage Archive

Submit the `submission/` folder (file lists + data + `metadata/`) to the BioImage Archive via
their FTP/Aspera/Globus transfer, following
[the BIA submission guide](https://www.ebi.ac.uk/bioimage-archive/help-file-list/).

Once published you receive an accession (`S-BIAD###`) and a DOI. Feed those back into the
config (`config.dataset.source_dataset_id` / `source_dataset_url`) and re-attach it to OMERO,
so a later BioImage Model Zoo export can cite this dataset as the model's `training_data`.

## 10. Cleanup

In [ ]:
if conn is not None:
    conn.close()
    print("OMERO connection closed.")